In [1]:
import glob, numpy as np
for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
    print(p)
for p in sorted(glob.glob("/kaggle/input/**/meta_test.npz", recursive=True)):
    m = np.load(p, allow_pickle=True)
    print(p)
    for k in m.files:
        print("  ", k, m[k].shape, m[k].dtype, m[k][:3])
for n in ["X_test.npy", "Y_test.npy"]:
    for p in glob.glob(f"/kaggle/input/**/{n}", recursive=True):
        a = np.load(p, mmap_mode="r")
        print(p, a.shape, a.dtype)
for pat in ["*.csv", "*.geojson", "*.shp", "*continent*"]:
    print(pat, glob.glob("/kaggle/input/**/" + pat, recursive=True)[:5])

/kaggle/input/notebooks/vinayybg/v2-03-train-e-1month/E_consensus_1month_best.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-1month/E_consensus_1month_last.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-consensus/E_consensus_best.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-consensus/E_consensus_last.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-dynworld/E_dynworld_best.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-dynworld/E_dynworld_last.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-worldcover/E_worldcover_best.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e-worldcover/E_worldcover_last.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e3-deeplab/E3_consensus_deeplab_best.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e3-deeplab/E3_consensus_deeplab_last.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e3-unet/E3_consensus_unet_best.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e3-unet/E3_consensus_unet_last.pt
/kaggle/input/notebooks/vinayybg/v2-03-train-e4-

In [2]:
import glob, json, os, time
import numpy as np, pandas as pd, torch
try:
    import segmentation_models_pytorch as smp
except ImportError:
    os.system("pip install -q segmentation-models-pytorch==0.6.0")
    import segmentation_models_pytorch as smp
dev, OUT = "cuda", "/kaggle/working"
CLS9 = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]
CLS6 = ["water", "trees", "low_veg", "built", "bare", "snow"]
L96 = [0, 1, 2, 2, 2, 2, 3, 4, 5]

def find(n):
    r = sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True))
    assert r, "not found: " + n
    return r[0]

MODELS = {
    "E_consensus": ("unet", "resnet18", 3, "E_consensus_best.pt"),
    "E_worldcover": ("unet", "resnet18", 3, "E_worldcover_best.pt"),
    "E_dynworld": ("unet", "resnet18", 3, "E_dynworld_best.pt"),
    "E_consensus_1month": ("unet", "resnet18", 1, "E_consensus_1month_best.pt"),
    "E3_unet_scratch": ("unet", "resnet18", 3, "E3_consensus_unet_best.pt"),
    "E3_deeplab_scratch": ("deeplab", "resnet18", 3, "E3_consensus_deeplab_best.pt"),
    "E4_unet_pre": ("unet", "resnet18", 3, "E4_consensus_unet_pre_best.pt"),
    "E4_deeplab_pre": ("deeplab", "resnet18", 3, "E4_consensus_deeplab_pre_best.pt"),
    "E5_fpn_r50_pre": ("fpn", "resnet50", 3, "E5_consensus_fpn_r50_best.pt"),
}
NETS = {"unet": smp.Unet, "deeplab": smp.DeepLabV3Plus, "fpn": smp.FPN}
norm = json.load(open(find("s1_norm.json")))
mean = torch.tensor(norm["mean"] * 3, dtype=torch.float32, device=dev).view(1, 6, 1, 1)
std = torch.tensor(norm["std"] * 3, dtype=torch.float32, device=dev).view(1, 6, 1, 1)
nets = {}
for name, (arch, enc, st, f) in MODELS.items():
    m = NETS[arch](enc, encoder_weights=None, in_channels=6 if st == 3 else 2, classes=9).to(dev)
    m.load_state_dict(torch.load(find(f), map_location=dev))
    nets[name] = m.eval()
    print("loaded", name)

Xm, Ym = np.load(find("X_test.npy"), mmap_mode="r"), np.load(find("Y_test.npy"), mmap_mode="r")
meta = np.load(find("meta_test.npz"))
vi = np.where(meta["valid"])[0]
uniq = np.unique(meta["pid"][vi])
loc = {p: i for i, p in enumerate(uniq.tolist())}
pid_loc = np.array([loc[p] for p in meta["pid"][vi].tolist()])
nP = len(uniq)
print("test patches:", len(vi), "| places:", nP)

cp = {m: torch.zeros(nP * 81, dtype=torch.long, device=dev) for m in MODELS}
cr = {(m, r): torch.zeros(81, dtype=torch.long, device=dev) for m in MODELS for r in ("worldcover", "dynworld")}
t0, B = time.time(), 64
with torch.no_grad():
    for s in range(0, len(vi), B):
        ii = vi[s:s + B]
        x = (torch.from_numpy(np.asarray(Xm[ii], dtype=np.float32)).to(dev) - mean) / std
        y = torch.from_numpy(np.asarray(Ym[ii])).to(dev).long()
        pl = torch.from_numpy(pid_loc[s:s + B]).to(dev).long().view(-1, 1, 1)
        for name, (arch, enc, st, f) in MODELS.items():
            with torch.autocast("cuda", dtype=torch.float16):
                out = nets[name](x if st == 3 else x[:, 2:4])
            p = out.argmax(1)
            yc = y[:, 0]
            mk = yc < 9
            cp[name] += torch.bincount((pl * 81 + yc * 9 + p)[mk], minlength=nP * 81)
            for r, li in (("worldcover", 1), ("dynworld", 2)):
                yy = y[:, li]
                mk = yy < 9
                cr[(name, r)] += torch.bincount((yy * 9 + p)[mk], minlength=81)
        if (s // B) % 15 == 0:
            print(f"{s}/{len(vi)} {time.time() - t0:.0f}s", flush=True)

def metrics(c):
    d, row, col = np.diag(c), c.sum(1), c.sum(0)
    iou = d / np.maximum(row + col - d, 1)
    f1 = 2 * d / np.maximum(row + col, 1)
    pres = row > 0
    return d.sum() / c.sum(), iou[pres].mean(), f1[pres].mean(), iou, f1

def merge6(c):
    M = np.zeros((6, 6))
    for i in range(9):
        for j in range(9):
            M[L96[i], L96[j]] += c[i, j]
    return M

Cp = {m: cp[m].view(nP, 9, 9).cpu().numpy().astype(np.float64) for m in MODELS}
C9 = {m: Cp[m].sum(0) for m in MODELS}
rng = np.random.default_rng(0)
idx = rng.integers(0, nP, (1000, nP))
boot = {}
for m in MODELS:
    boot[m] = np.array([metrics(Cp[m][i].sum(0))[:2] for i in idx])

rows = []
for m in MODELS:
    oa, mi, mf, iou, f1 = metrics(C9[m])
    lo, hi = np.percentile(boot[m][:, 1], [2.5, 97.5])
    olo, ohi = np.percentile(boot[m][:, 0], [2.5, 97.5])
    r = dict(model=m, OA=oa, OA_lo=olo, OA_hi=ohi, mIoU=mi, mIoU_lo=lo, mIoU_hi=hi, mF1=mf)
    r.update({f"IoU_{n}": v for n, v in zip(CLS9, iou)})
    r.update({f"F1_{n}": v for n, v in zip(CLS9, f1)})
    rows.append(r)
t9 = pd.DataFrame(rows).sort_values("mIoU", ascending=False)
t9.to_csv(f"{OUT}/test_9class_consensus.csv", index=False)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
print("\n=== 9 classes vs consensus, 95% CI by bootstrap over places ===")
print(t9[["model", "OA", "OA_lo", "OA_hi", "mIoU", "mIoU_lo", "mIoU_hi", "mF1"]].round(3).to_string(index=False))
print("\n=== per-class IoU (consensus) ===")
print(t9[["model"] + [f"IoU_{n}" for n in CLS9]].round(3).to_string(index=False))
print("\n=== per-class F1 (consensus) ===")
print(t9[["model"] + [f"F1_{n}" for n in CLS9]].round(3).to_string(index=False))

print("\n=== paired difference in mIoU vs E4_unet_pre (95% CI over places) ===")
for m in MODELS:
    if m != "E4_unet_pre":
        d = boot["E4_unet_pre"][:, 1] - boot[m][:, 1]
        lo, hi = np.percentile(d, [2.5, 97.5])
        print(f"{m:22s} diff {d.mean():+.3f}  [{lo:+.3f}, {hi:+.3f}]")

gt = C9["E4_unet_pre"].sum(1)
maj = int(gt.argmax())
print(f"\nmajority baseline: always '{CLS9[maj]}' -> OA {gt[maj] / gt.sum():.3f}, mIoU {gt[maj] / gt.sum() / (gt > 0).sum():.3f}")

for r in ("worldcover", "dynworld"):
    print(f"\n=== 9 classes vs {r} ===")
    rr = []
    for m in MODELS:
        oa, mi, mf, _, _ = metrics(cr[(m, r)].view(9, 9).cpu().numpy().astype(np.float64))
        rr.append((m, oa, mi, mf))
    print(pd.DataFrame(rr, columns=["model", "OA", "mIoU", "mF1"]).round(3).to_string(index=False))

print("\n=== 6 classes vs consensus ===")
r6 = []
for m in MODELS:
    oa, mi, mf, iou, _ = metrics(merge6(C9[m]))
    r6.append(dict(model=m, OA=oa, mIoU=mi, mF1=mf, **{f"IoU_{n}": v for n, v in zip(CLS6, iou)}))
print(pd.DataFrame(r6).round(3).to_string(index=False))

c = C9["E4_unet_pre"]
pairs = [(c[i, j] / c.sum(), i, j, c[i, j] / c[i].sum()) for i in range(9) for j in range(9) if i != j]
print("\n=== most confused pairs, E4_unet_pre vs consensus (share of all pixels | share of the true class) ===")
for v, i, j, rs in sorted(pairs, reverse=True)[:8]:
    print(f"true {CLS9[i]:8s} -> predicted {CLS9[j]:8s}: {100 * v:5.2f}% of pixels | {100 * rs:5.1f}% of {CLS9[i]}")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
cn = c / np.maximum(c.sum(1, keepdims=True), 1)
fig, a = plt.subplots(figsize=(8, 7))
a.imshow(cn, vmin=0, vmax=1, cmap="Blues")
a.set_title("E4_unet_pre vs consensus (row-normalised)")
a.set_xticks(range(9))
a.set_xticklabels(CLS9, rotation=45)
a.set_yticks(range(9))
a.set_yticklabels(CLS9)
a.set_xlabel("predicted")
a.set_ylabel("reference")
for i in range(9):
    for j in range(9):
        a.text(j, i, f"{cn[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT}/test_confusion_E4_unet_pre.png", dpi=120)
np.savez(f"{OUT}/test_confusions_per_place.npz", pids=uniq, **{m: Cp[m] for m in MODELS})
print("\nfiles:", sorted(os.listdir(OUT)))
print("ALL DONE")

loaded E_consensus
loaded E_worldcover
loaded E_dynworld
loaded E_consensus_1month
loaded E3_unet_scratch
loaded E3_deeplab_scratch
loaded E4_unet_pre
loaded E4_deeplab_pre
loaded E5_fpn_r50_pre
test patches: 4402 | places: 147
0/4402 9s
960/4402 31s
1920/4402 53s
2880/4402 76s
3840/4402 99s

=== 9 classes vs consensus, 95% CI by bootstrap over places ===
             model    OA  OA_lo  OA_hi  mIoU  mIoU_lo  mIoU_hi   mF1
    E5_fpn_r50_pre 0.811  0.779  0.839 0.630    0.582    0.657 0.755
       E4_unet_pre 0.799  0.767  0.829 0.623    0.572    0.657 0.752
    E4_deeplab_pre 0.801  0.769  0.831 0.620    0.563    0.656 0.750
E3_deeplab_scratch 0.795  0.763  0.824 0.609    0.560    0.642 0.741
   E3_unet_scratch 0.781  0.749  0.811 0.593    0.544    0.629 0.728
       E_consensus 0.778  0.745  0.808 0.586    0.539    0.619 0.721
      E_worldcover 0.758  0.725  0.791 0.555    0.506    0.588 0.691
E_consensus_1month 0.699  0.664  0.731 0.499    0.453    0.532 0.646
        E_dynworld 0.

In [3]:
import glob, numpy as np, pandas as pd
for p in sorted(glob.glob("/kaggle/input/**/places*.csv", recursive=True)):
    df = pd.read_csv(p)
    print(p, df.shape, list(df.columns))
    print(df.head(3).to_string())
for p in sorted(glob.glob("/kaggle/input/notebooks/vinayybg/v2-01-consensus/*"))[:30]:
    print(p)
m = np.load(glob.glob("/kaggle/input/**/meta_test.npz", recursive=True)[0])
pids = sorted(set(m["pid"][m["valid"]].tolist()))
print("test pids:", len(pids), "min", pids[0], "max", pids[-1], "first 12", pids[:12])

/kaggle/input/datasets/vinayybg/lulc-meta/places_final_FROZEN.csv (1136, 8) ['id', 'name', 'lat', 'lon', 'continent', 'group', 'run', 'split']
   id        name    lat   lon continent group  run  split
0   1  Heidelberg  49.40  8.69    Europe  land    1  train
1   2       Milan  45.46  9.19    Europe  land    1  train
2   3    LochNess  57.30 -4.45    Europe  land    1  train
/kaggle/input/datasets/vinayybg/places-v2/places_v2.csv (1000, 7) ['id', 'name', 'lat', 'lon', 'continent', 'usable', 'split']
   id        name    lat   lon continent  usable  split
0   1  Heidelberg  49.40  8.69    Europe    True  train
1   2       Milan  45.46  9.19    Europe    True  train
2   3    LochNess  57.30 -4.45    Europe    True  train
/kaggle/input/datasets/vinayybg/run9-plan/places_final_FROZEN.csv (1136, 8) ['id', 'name', 'lat', 'lon', 'continent', 'group', 'run', 'split']
   id        name    lat   lon continent group  run  split
0   1  Heidelberg  49.40  8.69    Europe  land    1  train
1   2    

In [4]:
import os, glob, numpy as np, pandas as pd
pl = pd.read_csv("/kaggle/input/notebooks/vinayybg/sar-lulc-run01/places.csv")
print("places.csv ids:", pl.id.min(), "to", pl.id.max(), "| rows", len(pl))
m = np.load(glob.glob("/kaggle/input/**/meta_test.npz", recursive=True)[0])
pids = sorted(set(m["pid"][m["valid"]].tolist()))
low = [p for p in pids if p <= 1000]
print("test pids <= 1000:", len(low), "| split of those places in places.csv:", pl[pl.id.isin(low)].split.value_counts().to_dict(), "| not in file:", len(set(low) - set(pl.id)))
print("test pids > 1000:", [p for p in pids if p > 1000])
for run in ["sar-lulc-run09", "sar-lulc-run01"]:
    d = f"/kaggle/input/notebooks/vinayybg/{run}"
    print(run, sorted(os.listdir(d))[:20])

places.csv ids: 1 to 1000 | rows 1000
test pids <= 1000: 129 | split of those places in places.csv: {'train': 106, 'test': 17, 'val': 6} | not in file: 0
test pids > 1000: [1011, 1024, 1034, 1036, 1041, 1044, 1047, 1053, 1070, 1074, 1082, 1091, 1096, 1104, 1111, 1114, 1116, 1139]
sar-lulc-run09 ['__notebook__.ipynb', '__output__.json', '__results__.html', 'custom.css', 'run9']
sar-lulc-run01 ['__notebook__.ipynb', '__output__.json', '__results__.html', 'custom.css', 'places.csv', 'run1']


In [5]:
import os, glob
d = "/kaggle/input/notebooks/vinayybg/v2-01-consensus"
for p in sorted(glob.glob(d + "/**/*", recursive=True))[:60]:
    print(p, round(os.path.getsize(p) / 1e6, 2), "MB" if os.path.isfile(p) else "(dir)")

/kaggle/input/notebooks/vinayybg/v2-01-consensus/__notebook__.ipynb 0.01 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/__output__.json 0.0 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/__results__.html 0.31 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons 0.0 (dir)
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0001.npz 0.04 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0002.npz 0.01 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0003.npz 0.08 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0004.npz 0.11 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0005.npz 0.09 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0006.npz 0.04 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0007.npz 0.01 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0008.npz 0.01 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0009.npz 0.05 MB
/kaggle/input/notebooks/vinayybg/v2-01-consensus/cons/0010.npz 0.05 MB
/kag

In [6]:
import os, numpy as np
d = "/kaggle/input/notebooks/vinayybg/v2-01-consensus"
print(sorted(os.listdir(d)))
fs = sorted(os.listdir(d + "/cons"))
print(len(fs), "label files | first", fs[:2], "| last", fs[-3:])
z = np.load(d + "/cons/0026.npz")
print(z.files)
for k in z.files:
    print(k, z[k].shape, z[k].dtype)

['__notebook__.ipynb', '__output__.json', '__results__.html', 'cons', 'custom.css', 's1_norm.json', 'v2_places.csv']
1124 label files | first ['0001.npz', '0002.npz'] | last ['1141.npz', '1142.npz', '1143.npz']
['cons']
cons (9, 2, 256, 256) uint8


In [7]:
import glob, numpy as np, pandas as pd
v2 = pd.read_csv("/kaggle/input/notebooks/vinayybg/v2-01-consensus/v2_places.csv")
print(v2.shape, list(v2.columns))
print(v2.head(4).to_string())
for c in v2.columns:
    if v2[c].nunique() <= 6:
        print(c, v2[c].value_counts().to_dict())
m = np.load(glob.glob("/kaggle/input/**/meta_test.npz", recursive=True)[0])
pids = sorted(set(m["pid"][m["valid"]].tolist()))
idcol = [c for c in v2.columns if c.lower() in ("id", "pid", "place_id")]
print("id column candidates:", idcol)
if idcol:
    sub = v2[v2[idcol[0]].isin(pids)]
    print("test pids found in v2_places:", len(sub), "of", len(pids))
    for c in v2.columns:
        if v2[c].nunique() <= 6:
            print("among test places", c, sub[c].value_counts().to_dict())

(1124, 5) ['id', 'run', 'split', 'patches', 'months_ok']
   id      run  split  patches  months_ok
0   1  v2_run1  train        9         24
1   2  v2_run1  train        9         24
2   3  v2_run1  train       12         24
3   4  v2_run1  train        9         24
run {'v2_run1': 225, 'v2_run2': 225, 'v2_run3': 225, 'v2_run4': 225, 'v2_run5': 224}
split {'train': 894, 'test': 148, 'val': 82}
patches {9: 863, 16: 208, 12: 53}
id column candidates: ['id']
test pids found in v2_places: 147 of 147
among test places run {'v2_run2': 38, 'v2_run3': 32, 'v2_run5': 29, 'v2_run1': 25, 'v2_run4': 23}
among test places split {'test': 147}
among test places patches {9: 111, 16: 29, 12: 7}


In [8]:
import glob, os
base = "/kaggle/input/notebooks/vinayybg"
print("csv files at top level:", glob.glob(base + "/*/*.csv"))
r9 = base + "/sar-lulc-run09"
others = [p for p in glob.glob(r9 + "/**/*", recursive=True) if os.path.isfile(p) and not p.endswith(".npz")]
print("non-npz files in run09:", others[:20])
ids9 = sorted(os.listdir(r9 + "/run9"))
print("run9 folders:", len(ids9), ids9[:5], ids9[-3:])

csv files at top level: ['/kaggle/input/notebooks/vinayybg/v2-01-consensus/v2_places.csv', '/kaggle/input/notebooks/vinayybg/sar-lulc-run01/places.csv']
non-npz files in run09: ['/kaggle/input/notebooks/vinayybg/sar-lulc-run09/__notebook__.ipynb', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/__output__.json', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/__results__.html', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/custom.css', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1001/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1002/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1003/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1004/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1005/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1006/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1007/DONE', '/kaggle/input/notebooks/vinayybg/sar-lulc-run09/run9/1008/DONE', '/kaggle/input/notebooks/vi

In [9]:
import json
nb = json.load(open("/kaggle/input/notebooks/vinayybg/sar-lulc-run09/__notebook__.ipynb"))
cells = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
print(len(cells), "code cells, lengths", [len(c) for c in cells])
for i, c in enumerate(cells):
    if "lat" in c.lower() or "place" in c.lower():
        print(f"--- cell {i} ---")
        print(c[:3500])

3 code cells, lengths [733, 71, 2637]
--- cell 0 ---
import ee, json, os, math, io, time, glob, requests
import numpy as np, pandas as pd, rasterio
from concurrent.futures import ThreadPoolExecutor
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("EE_KEY")
email = json.loads(key)["client_email"]
ee.Initialize(ee.ServiceAccountCredentials(email, key_data=key), project="lulc-sar-project")

RUN = 9
START_Y, START_M, N_MONTHS = 2025, 1, 21
KM, P = 10, 256
OUT = f"/kaggle/working/run{RUN}"
os.makedirs(OUT, exist_ok=True)

pf = glob.glob("/kaggle/input/**/run9_places.csv", recursive=True)[0]
places = pd.read_csv(pf)
print("places in this run:", len(places), "| ids", places.id.min(), "-", places.id.max())
print(places.split.value_counts().to_dict())
--- cell 2 ---
def utm(lat, lon):
    return f"EPSG:{(32600 if lat >= 0 else 32700) + int((lon + 180)//6) + 1}"

def box(lat, lon):
    dlat = KM/2/111.32
    dlon = KM/2/(111.32*math.cos(math.radians(lat)))
    r

In [10]:
import glob, pandas as pd
for ds in ["run9-plan", "places-v2", "lulc-meta"]:
    for p in sorted(glob.glob(f"/kaggle/input/**/{ds}/*.csv", recursive=True)) or sorted(glob.glob(f"/kaggle/input/{ds}/**/*.csv", recursive=True)):
        df = pd.read_csv(p)
        print(p, df.shape, list(df.columns))
        print(df.head(2).to_string())

/kaggle/input/datasets/vinayybg/run9-plan/cities_final.csv (41, 14) ['id', 'geonameid', 'name', 'ADMIN', 'CONTINENT', 'lat', 'lon', 'pop', 'size', 'band', 'share', 'min_km', 'continent_fixed', 'group']
     id  geonameid          name     ADMIN CONTINENT       lat        lon      pop   size band     share      min_km continent_fixed group
0  1001    1785725         Yunfu     China      Asia  22.92787  112.03809  2612800  large  low  0.220641  107.258775            Asia  city
1  1002    1608527  Nakhon Sawan  Thailand      Asia  15.70472  100.13717    82305  small  low  0.271672  167.586357            Asia  city
/kaggle/input/datasets/vinayybg/run9-plan/coastal_selected.csv (30, 8) ['id', 'lat', 'lon', 'continent', 'band', 'landshare', 'min_km', 'group']
     id   lat    lon continent    band  landshare    min_km    group
0  1072  -7.5  128.5      Asia  tropic     0.3020  836.1128  coastal
1  1073  42.5  132.5      Asia     mid     0.4682  241.0211  coastal
/kaggle/input/datasets/vinayy

In [11]:
import glob, json, os, time
import numpy as np, pandas as pd, torch
try:
    import segmentation_models_pytorch as smp
except ImportError:
    os.system("pip install -q segmentation-models-pytorch==0.6.0")
    import segmentation_models_pytorch as smp
dev, OUT = "cuda", "/kaggle/working"
CLS9 = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]

def find(n):
    r = sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True))
    assert r, "not found: " + n
    return r[0]

MODELS = {
    "E3_unet_scratch": ("unet", "resnet18", "E3_consensus_unet_best.pt"),
    "E3_deeplab_scratch": ("deeplab", "resnet18", "E3_consensus_deeplab_best.pt"),
    "E4_unet_pre": ("unet", "resnet18", "E4_consensus_unet_pre_best.pt"),
    "E4_deeplab_pre": ("deeplab", "resnet18", "E4_consensus_deeplab_pre_best.pt"),
    "E5_fpn_r50_pre": ("fpn", "resnet50", "E5_consensus_fpn_r50_best.pt"),
}
NETS = {"unet": smp.Unet, "deeplab": smp.DeepLabV3Plus, "fpn": smp.FPN}
norm = json.load(open(find("s1_norm.json")))
mean = torch.tensor(norm["mean"] * 3, dtype=torch.float32, device=dev).view(1, 6, 1, 1)
std = torch.tensor(norm["std"] * 3, dtype=torch.float32, device=dev).view(1, 6, 1, 1)
nets = {}
for name, (arch, enc, f) in MODELS.items():
    m = NETS[arch](enc, encoder_weights=None, in_channels=6, classes=9).to(dev)
    m.load_state_dict(torch.load(find(f), map_location=dev))
    nets[name] = m.eval()

Xm, Ym = np.load(find("X_test.npy"), mmap_mode="r"), np.load(find("Y_test.npy"), mmap_mode="r")
meta = np.load(find("meta_test.npz"))
vi = np.where(meta["valid"])[0]
uniq = np.unique(meta["pid"][vi])
loc = {p: i for i, p in enumerate(uniq.tolist())}
pid_loc = np.array([loc[p] for p in meta["pid"][vi].tolist()])
nP = len(uniq)
n_patch = np.bincount(pid_loc, minlength=nP)
cp = {m: torch.zeros(nP * 81, dtype=torch.long, device=dev) for m in MODELS}
t0, B = time.time(), 64
with torch.no_grad():
    for s in range(0, len(vi), B):
        ii = vi[s:s + B]
        x = (torch.from_numpy(np.asarray(Xm[ii], dtype=np.float32)).to(dev) - mean) / std
        yc = torch.from_numpy(np.asarray(Ym[ii][:, 0])).to(dev).long()
        pl_ = torch.from_numpy(pid_loc[s:s + B]).to(dev).long().view(-1, 1, 1)
        for name in MODELS:
            with torch.autocast("cuda", dtype=torch.float16):
                p = nets[name](x).argmax(1)
            mk = yc < 9
            cp[name] += torch.bincount((pl_ * 81 + yc * 9 + p)[mk], minlength=nP * 81)
print("scored in", round(time.time() - t0), "s")
Cp = {m: cp[m].view(nP, 9, 9).cpu().numpy().astype(np.float64) for m in MODELS}

info = pd.read_csv(find("places_final_FROZEN.csv"), keep_default_na=False).set_index("id")
missing = [p for p in uniq.tolist() if p not in info.index]
print("test places without coordinates:", len(missing), missing[:10])
cont = np.array([info.loc[p, "continent"] if p in info.index else "unknown" for p in uniq])
grp = np.array([info.loc[p, "group"] if p in info.index else "unknown" for p in uniq])

def metrics(c):
    d, row, col = np.diag(c), c.sum(1), c.sum(0)
    iou = d / np.maximum(row + col - d, 1)
    return d.sum() / max(c.sum(), 1), iou[row > 0].mean()

rng = np.random.default_rng(0)
def table(labels, title):
    rows = []
    for g in sorted(set(labels)):
        ix = np.where(labels == g)[0]
        r = dict(group=g, places=len(ix), patches=int(n_patch[ix].sum()))
        for m in MODELS:
            oa, mi = metrics(Cp[m][ix].sum(0))
            r[m + "_mIoU"] = round(mi, 3)
        oa, mi = metrics(Cp["E4_unet_pre"][ix].sum(0))
        bs = [metrics(Cp["E4_unet_pre"][ix[rng.integers(0, len(ix), len(ix))]].sum(0))[1] for _ in range(500)]
        r["E4_unet_OA"] = round(oa, 3)
        r["E4_unet_mIoU_lo"], r["E4_unet_mIoU_hi"] = np.round(np.percentile(bs, [2.5, 97.5]), 3)
        rows.append(r)
    df = pd.DataFrame(rows)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    print(f"\n=== by {title} (mIoU vs consensus; CI for E4_unet_pre by bootstrap over places) ===")
    print(df.to_string(index=False))
    return df

t_c = table(cont, "continent")
t_g = table(grp, "place type")
t_c.to_csv(f"{OUT}/test_by_continent.csv", index=False)
t_g.to_csv(f"{OUT}/test_by_group.csv", index=False)
print("ALL DONE")

scored in 52 s
test places without coordinates: 0 []

=== by continent (mIoU vs consensus; CI for E4_unet_pre by bootstrap over places) ===
        group  places  patches  E3_unet_scratch_mIoU  E3_deeplab_scratch_mIoU  E4_unet_pre_mIoU  E4_deeplab_pre_mIoU  E5_fpn_r50_pre_mIoU  E4_unet_OA  E4_unet_mIoU_lo  E4_unet_mIoU_hi
                   18      540                 0.461                    0.460             0.479                0.507                0.465       0.820            0.406            0.510
       Africa      32      952                 0.467                    0.493             0.545                0.510                0.551       0.825            0.401            0.590
         Asia      39     1170                 0.463                    0.484             0.503                0.476                0.501       0.816            0.412            0.543
       Europe      12      360                 0.416                    0.422             0.444                0.420        